In [ ]:
from pathlib import Path
import re
import time
import pandas as pd
import requests
from tqdm import tqdm


# =========================================================
# 1. 文件设置
# =========================================================

DESKTOP = Path("/Users/tanglikun/Desktop")

# 会自动优先读取第三轮文件；如果你文件名不一样，就手动改这里
INPUT_FILE = DESKTOP / "A股AI产业链召回总表_第三轮.xlsx"

OUTPUT_FILE = DESKTOP / "A股AI产业链召回总表_第四轮_补充公司文本.xlsx"

# 先测试可以改成 50；正式跑全量改成 None
MAX_COMPANIES = None

# 每家公司之间暂停，避免访问太快
SLEEP_SECONDS = 0.6

# 每处理多少家公司保存一次
SAVE_EVERY = 50


# =========================================================
# 2. 第四轮 recall 关键词
# 这一轮基于“公司简介/主营业务/经营范围/公告标题”继续筛
# =========================================================

FOURTH_LAYER_KEYWORDS = {
    "AI综合": [
        "人工智能", "AI", "AIGC", "生成式AI", "大模型", "多模态",
        "ChatGPT", "机器学习", "深度学习", "神经网络", "智能算法"
    ],
    "AI基础层-算力": [
        "算力", "智算", "智能算力", "智算中心", "算力中心",
        "数据中心", "IDC", "云计算", "边缘计算", "超算",
        "高性能计算", "AI服务器", "服务器", "液冷服务器"
    ],
    "AI基础层-芯片半导体": [
        "AI芯片", "GPU", "GPGPU", "FPGA", "ASIC", "NPU", "DPU",
        "SoC", "推理芯片", "训练芯片", "算力芯片", "存储芯片",
        "HBM", "先进封装", "Chiplet", "半导体设备", "EDA"
    ],
    "AI基础层-光通信网络": [
        "光模块", "CPO", "硅光", "光通信", "光芯片", "光器件",
        "交换机", "路由器", "网络设备", "800G", "1.6T",
        "高速连接器", "高速线缆"
    ],
    "AI基础层-PCB散热电源": [
        "PCB", "高速PCB", "HDI", "封装基板", "覆铜板",
        "液冷", "散热", "温控", "机柜", "UPS", "电源模块"
    ],
    "模型与软件层": [
        "自然语言处理", "NLP", "计算机视觉", "机器视觉",
        "语音识别", "语音合成", "知识图谱", "数据标注",
        "数据治理", "数据要素", "模型训练", "模型推理",
        "AI平台", "MaaS", "智能决策", "OCR", "RPA"
    ],
    "AI应用-机器人": [
        "机器人", "工业机器人", "服务机器人", "人形机器人",
        "协作机器人", "移动机器人", "AGV", "AMR",
        "伺服系统", "减速器", "控制器", "运动控制",
        "灵巧手", "执行器"
    ],
    "AI应用-智能汽车": [
        "自动驾驶", "智能驾驶", "辅助驾驶", "ADAS",
        "智能座舱", "车路协同", "智能网联", "域控制器",
        "激光雷达", "毫米波雷达", "高精地图", "线控底盘"
    ],
    "AI应用-行业场景": [
        "智能制造", "工业互联网", "智慧工厂", "智慧医疗",
        "医学影像", "智能诊断", "AI制药", "智慧金融",
        "智能客服", "智慧城市", "智能安防", "智慧交通",
        "智慧能源", "智慧矿山", "智慧物流", "智能仓储"
    ],
    "AI终端与交互": [
        "智能终端", "智能穿戴", "智能家居", "AR", "VR", "MR",
        "XR", "空间计算", "智能音箱", "智能语音",
        "数字人", "虚拟人", "无人机"
    ],
}

WEAK_KEYWORDS = [
    "智能化", "数字化", "自动化", "信息化", "系统集成",
    "解决方案", "平台建设", "数据平台", "云平台"
]


# =========================================================
# 3. 工具函数
# =========================================================

def normalize_code(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.replace("bj", "").replace("sh", "").replace("sz", "")
    s = re.sub(r"\D", "", s)
    if len(s) > 6:
        s = s[-6:]
    return s.zfill(6) if s else ""


def find_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    for col in df.columns:
        for c in candidates:
            if c in str(col):
                return col
    return None


def guess_cninfo_column(code):
    if code.startswith(("600", "601", "603", "605", "688", "689")):
        return "sse"
    if code.startswith(("000", "001", "002", "003", "300", "301")):
        return "szse"
    if code.startswith(("430", "831", "832", "833", "834", "835", "836", "837", "838", "839", "870", "871", "872", "873", "920")):
        return "bj"
    return "szse"


def append_text(old, new):
    old = "" if pd.isna(old) else str(old)
    new = "" if pd.isna(new) else str(new)
    if not old:
        return new
    if not new:
        return old
    if new in old:
        return old
    return old + "；" + new


def find_hits_by_layer(text, keyword_dict):
    text = str(text)
    hit_keywords = []
    hit_layers = []

    for layer, keywords in keyword_dict.items():
        for kw in keywords:
            if kw in text:
                hit_keywords.append(kw)
                hit_layers.append(layer)

    return list(dict.fromkeys(hit_keywords)), list(dict.fromkeys(hit_layers))


def find_hits(text, keywords):
    text = str(text)
    return [kw for kw in keywords if kw in text]


def get_hit_text(text, hits, max_len=260):
    if not hits:
        return ""

    text = str(text).replace("\n", " ").replace("\r", " ")

    for kw in hits:
        pos = text.find(kw)
        if pos != -1:
            start = max(pos - 100, 0)
            end = min(pos + 160, len(text))
            return text[start:end][:max_len]

    return ""


def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


# =========================================================
# 4. AkShare 补公司资料
# =========================================================

def fetch_akshare_company_text(code):
    """
    尝试用 AkShare 补充公司资料。
    有些接口会失败，所以这里全部安全处理，失败就留空。
    """
    info = {
        "公司简介": "",
        "主营业务": "",
        "经营范围": "",
        "所属行业补充": "",
        "概念标签": "",
        "补充文本来源": []
    }

    try:
        import akshare as ak
    except Exception:
        return info

    # 1. 东方财富个股信息：有时只包含行业、市值等
    try:
        df_info = ak.stock_individual_info_em(symbol=code)
        if isinstance(df_info, pd.DataFrame) and not df_info.empty:
            text = " ".join(df_info.astype(str).fillna("").values.flatten().tolist())
            if "行业" in text or "行业：" in text:
                info["所属行业补充"] = text[:300]
            info["公司简介"] = append_text(info["公司简介"], text[:500])
            info["补充文本来源"].append("AkShare东方财富个股信息")
    except Exception:
        pass

    # 2. 巨潮公司简介接口，如果当前 AkShare 版本支持
    try:
        if hasattr(ak, "stock_profile_cninfo"):
            df_profile = ak.stock_profile_cninfo(symbol=code)
            if isinstance(df_profile, pd.DataFrame) and not df_profile.empty:
                text = " ".join(df_profile.astype(str).fillna("").values.flatten().tolist())
                info["公司简介"] = append_text(info["公司简介"], text[:800])

                for col in df_profile.columns:
                    col_s = str(col)
                    if "经营范围" in col_s:
                        info["经营范围"] = append_text(info["经营范围"], " ".join(df_profile[col].astype(str).tolist())[:800])
                    if "主营" in col_s:
                        info["主营业务"] = append_text(info["主营业务"], " ".join(df_profile[col].astype(str).tolist())[:800])
                    if "行业" in col_s:
                        info["所属行业补充"] = append_text(info["所属行业补充"], " ".join(df_profile[col].astype(str).tolist())[:300])

                info["补充文本来源"].append("AkShare巨潮公司简介")
    except Exception:
        pass

    # 3. 同花顺主营介绍，如果当前 AkShare 版本支持
    try:
        if hasattr(ak, "stock_zyjs_ths"):
            df_zy = ak.stock_zyjs_ths(symbol=code)
            if isinstance(df_zy, pd.DataFrame) and not df_zy.empty:
                text = " ".join(df_zy.astype(str).fillna("").values.flatten().tolist())
                info["主营业务"] = append_text(info["主营业务"], text[:1000])
                info["补充文本来源"].append("AkShare同花顺主营介绍")
    except Exception:
        pass

    info["补充文本来源"] = "、".join(list(dict.fromkeys(info["补充文本来源"])))
    return info


# =========================================================
# 5. 巨潮补最近公告标题
# =========================================================

def fetch_cninfo_ann_titles(code, max_titles=20):
    url = "https://www.cninfo.com.cn/new/hisAnnouncement/query"

    column = guess_cninfo_column(code)

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.cninfo.com.cn/new/commonUrl/pageOfSearch?url=disclosure/list/search",
        "Accept": "application/json, text/javascript, */*; q=0.01",
    }

    data = {
        "stock": code,
        "searchkey": "",
        "plate": "",
        "category": "",
        "trade": "",
        "column": column,
        "pageNum": 1,
        "pageSize": max_titles,
        "tabName": "fulltext",
        "sortName": "",
        "sortType": "",
        "limit": "",
        "seDate": "2023-01-01~2026-12-31",
    }

    try:
        r = requests.post(url, headers=headers, data=data, timeout=20)
        if r.status_code != 200:
            return ""

        js = r.json()
        anns = js.get("announcements", []) or []

        titles = []
        for ann in anns:
            title = ann.get("announcementTitle", "")
            title = re.sub(r"<.*?>", "", title)
            title = title.strip()
            if title:
                titles.append(title)

        return "；".join(titles[:max_titles])

    except Exception:
        return ""


# =========================================================
# 6. 主程序
# =========================================================

def main():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"没有找到输入文件：{INPUT_FILE}")

    df = pd.read_excel(INPUT_FILE, dtype=str).fillna("")

    code_col = find_column(df, ["股票代码", "证券代码", "代码"])
    name_col = find_column(df, ["股票简称", "证券简称", "名称", "简称"])

    if code_col is None:
        raise ValueError("没有找到股票代码列")
    if name_col is None:
        raise ValueError("没有找到股票简称列")

    df["股票代码"] = df[code_col].apply(normalize_code)

    needed_cols = [
        "公司简介",
        "主营业务",
        "经营范围",
        "所属行业补充",
        "概念标签",
        "最近公告标题",
        "补充文本来源",
        "是否在AI产业链中",
        "召回来源",
        "命中关键词",
        "命中文本",
        "AI产业链环节",
        "确定性等级",
        "复核状态",
        "备注",
    ]

    for col in needed_cols:
        if col not in df.columns:
            df[col] = ""

    if MAX_COMPANIES is not None:
        work_index = df.index[:MAX_COMPANIES]
    else:
        work_index = df.index

    print(f"开始第四轮：补充公司文本，共处理 {len(work_index)} 家公司")

    step4_sources = []
    step4_keywords = []
    step4_texts = []
    step4_layers = []

    for n, i in enumerate(tqdm(work_index, total=len(work_index)), start=1):
        code = df.loc[i, "股票代码"]

        # 1. 补 AkShare 公司资料
        ak_info = fetch_akshare_company_text(code)

        df.loc[i, "公司简介"] = append_text(df.loc[i, "公司简介"], ak_info["公司简介"])
        df.loc[i, "主营业务"] = append_text(df.loc[i, "主营业务"], ak_info["主营业务"])
        df.loc[i, "经营范围"] = append_text(df.loc[i, "经营范围"], ak_info["经营范围"])
        df.loc[i, "所属行业补充"] = append_text(df.loc[i, "所属行业补充"], ak_info["所属行业补充"])
        df.loc[i, "概念标签"] = append_text(df.loc[i, "概念标签"], ak_info["概念标签"])
        df.loc[i, "补充文本来源"] = append_text(df.loc[i, "补充文本来源"], ak_info["补充文本来源"])

        # 2. 补巨潮最近公告标题
        ann_titles = fetch_cninfo_ann_titles(code)
        if ann_titles:
            df.loc[i, "最近公告标题"] = append_text(df.loc[i, "最近公告标题"], ann_titles)
            df.loc[i, "补充文本来源"] = append_text(df.loc[i, "补充文本来源"], "巨潮最近公告标题")

        # 3. 基于新增文本做第四轮 recall
        combined_text = " ".join([
            safe_str(df.loc[i, name_col]),
            safe_str(df.loc[i, "公司简介"]),
            safe_str(df.loc[i, "主营业务"]),
            safe_str(df.loc[i, "经营范围"]),
            safe_str(df.loc[i, "所属行业补充"]),
            safe_str(df.loc[i, "概念标签"]),
            safe_str(df.loc[i, "最近公告标题"]),
        ])

        strong_hits, layers = find_hits_by_layer(combined_text, FOURTH_LAYER_KEYWORDS)
        weak_hits = find_hits(combined_text, WEAK_KEYWORDS)

        if strong_hits:
            source = "第四轮补充公司文本召回"
            df.loc[i, "是否在AI产业链中"] = "是"
            df.loc[i, "召回来源"] = append_text(df.loc[i, "召回来源"], source)
            df.loc[i, "命中关键词"] = append_text(df.loc[i, "命中关键词"], "、".join(strong_hits))
            df.loc[i, "命中文本"] = append_text(df.loc[i, "命中文本"], get_hit_text(combined_text, strong_hits))
            df.loc[i, "AI产业链环节"] = append_text(df.loc[i, "AI产业链环节"], "、".join(layers))
            df.loc[i, "确定性等级"] = "中"
            df.loc[i, "复核状态"] = "待复核"
            df.loc[i, "备注"] = append_text(
                df.loc[i, "备注"],
                "第四轮基于公司简介/主营业务/经营范围/公告标题命中AI产业链关键词"
            )

        elif weak_hits and df.loc[i, "是否在AI产业链中"] != "是":
            source = "第四轮补充文本弱相关"
            df.loc[i, "是否在AI产业链中"] = "不能确定"
            df.loc[i, "召回来源"] = append_text(df.loc[i, "召回来源"], source)
            df.loc[i, "命中关键词"] = append_text(df.loc[i, "命中关键词"], "、".join(weak_hits))
            df.loc[i, "命中文本"] = append_text(df.loc[i, "命中文本"], get_hit_text(combined_text, weak_hits))
            df.loc[i, "确定性等级"] = "低"
            df.loc[i, "复核状态"] = "进入下一轮"
            df.loc[i, "备注"] = append_text(
                df.loc[i, "备注"],
                "第四轮仅命中弱相关词，需要进一步用年报全文或人工复核"
            )

        else:
            source = "第四轮未召回"

        step4_sources.append(source)
        step4_keywords.append("、".join(strong_hits + weak_hits))
        step4_texts.append(get_hit_text(combined_text, strong_hits + weak_hits))
        step4_layers.append("、".join(layers))

        if n % SAVE_EVERY == 0:
            df.to_excel(OUTPUT_FILE, index=False)
            print(f"已临时保存：{OUTPUT_FILE}")

        time.sleep(SLEEP_SECONDS)

    # 如果只处理部分公司，长度要补齐
    df["第四轮召回来源"] = ""
    df["第四轮命中关键词"] = ""
    df["第四轮命中文本"] = ""
    df["第四轮AI产业链环节"] = ""

    for idx, source, kws, txt, layers in zip(work_index, step4_sources, step4_keywords, step4_texts, step4_layers):
        df.loc[idx, "第四轮召回来源"] = source
        df.loc[idx, "第四轮命中关键词"] = kws
        df.loc[idx, "第四轮命中文本"] = txt
        df.loc[idx, "第四轮AI产业链环节"] = layers

    df.to_excel(OUTPUT_FILE, index=False)

    print("第四轮完成！")
    print(f"输出文件：{OUTPUT_FILE}")
    print()
    print("当前总判断结果：")
    print(df["是否在AI产业链中"].value_counts())
    print()
    print("第四轮召回情况：")
    print(df["第四轮召回来源"].value_counts())


if __name__ == "__main__":
    main()

开始第四轮：补充公司文本，共处理 5527 家公司


  1%|          | 49/5527 [25:21<154:19:27, 101.42s/it]

已临时保存：/Users/tanglikun/Desktop/A股AI产业链召回总表_第四轮_补充公司文本.xlsx


  2%|▏         | 99/5527 [26:54<2:30:36,  1.66s/it]   

已临时保存：/Users/tanglikun/Desktop/A股AI产业链召回总表_第四轮_补充公司文本.xlsx


...

 25%|██▌       | 1399/5527 [1:06:33<2:00:43,  1.75s/it]

已临时保存：/Users/tanglikun/Desktop/A股AI产业链召回总表_第四轮_补充公司文本.xlsx


 26%|██▌       | 1449/5527 [1:08:53<6:44:35,  5.95s/it]

已临时保存：/Users/tanglikun/Desktop/A股AI产业链召回总表_第四轮_补充公司文本.xlsx


 26%|██▋       | 1457/5527 [1:12:39<34:16:18, 30.31s/it]